# Problème de Poisson réel (avant d'aborder le complexe) sur un rectangle
On commence par importer des librairies qui nous serviront plus tard

In [1]:
from pathlib import Path

from mpi4py import MPI
from petsc4py.PETSc import ScalarType  # type: ignore

import numpy as np

import ufl
from dolfinx import fem, io, mesh, plot
from dolfinx.fem.petsc import LinearProblem


Définition du domaine Oméga - rectangle [a, b]x[c, d] pour l'instant, et des paramètres du pb

In [2]:
degre_elements_finis = 1
(a, c), (b, d) = (0.0, 0.0), (2.0, 1.0) # Dans la sectio suivante, mesh.create_rectangle prend en argument points
                                        # les coordonnées des points sur la première diagonale du rectangle
                                        # (en bas, à gauche) et (en haut, à droite)
(Nx, Ny) = (32, 16)

In [3]:
# Création d'un mesh
msh = mesh.create_rectangle(
    comm=MPI.COMM_WORLD,
    points=((a, c), (b, d)),
    n=(Nx, Ny),
    cell_type=mesh.CellType.triangle,
)

# et d'une classe d'éléments finis sur Oméga
V = fem.functionspace(msh, ("Lagrange", degre_elements_finis))

In [4]:
# Définition des frontières (facets)
facets = mesh.locate_entities_boundary(
    msh,
    dim=(msh.topology.dim - 1),
    marker=lambda x: np.isclose(x[0], a) | np.isclose(x[0], a),
)